In [39]:
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer

pd.set_option('display.max_rows', 50)

df_test = pd.read_csv('test_data.csv')
#df_test = df_test.iloc[:,1:]
df_test.head()
#df_test['ID 1'].value_counts()

,Grade,ID 1,Fe,C,Si,Mn,P,S,Cr,Mo,...,Ti,V,W,Pb,Sn,B,Ca,Zr,As,Bi
0,20_21_1,St 48,0.0,0.0331,0.204,0.701,0.0291,0.0295,0.0075,0.0051,...,0.0010,0.0020,0.04,0.0150,0.0080,0.001,0.0010,0.002,0.0507,0.0114
1,20_23_1,St 48,0.0,0.1620,0.130,0.720,0.0217,0.0411,0.0052,0.0041,...,0.0032,0.0018,0.04,0.0347,0.0122,0.001,0.0013,0.002,0.0629,0.0108
2,20_28_1,St 48,0.0,0.1440,0.265,0.600,0.0186,0.0261,0.0050,0.0049,...,0.0012,0.0018,0.04,0.0150,0.0191,0.001,0.0010,0.002,0.0638,0.0100
3,20_31_1,St 48,0.0,0.2640,0.274,0.627,0.0129,0.0312,0.0050,0.0049,...,0.0012,0.0023,0.04,0.0150,0.0089,0.001,0.0010,0.002,0.0353,0.0100
4,20_32_1,St 48,0.0,0.2550,0.288,0.643,0.0114,0.0577,0.0050,0.0040,...,0.0014,0.0023,0.04,0.0150,0.0106,0.001,0.0010,0.002,0.0444,0.0103


In [2]:
# Auswählen der Spalten, die skaliert werden sollen
columns_to_scale = ['Fe', 'C', 'Si', 'Mn', 'P', 'S', 'Cr', 'Mo', 'Ni', 'Al', 'Co', 'Cu', 'Nb', 'Ti', 'V', 'W', 'Pb', 'Sn', 'B', 'Ca', 'Zr', 'As', 'Bi']

# Entferne Zeilen, bei denen in 'ID 1' ein '?' vorkommt
# df_test = df_test.loc[~df_test['ID 1'].str.contains('\?', na=False)]

# Skalieren der ausgewählten Spalten
scaler = FunctionTransformer(np.log1p, validate=True)
scaled_data = scaler.fit_transform(df_test[columns_to_scale])

# Erstellen eines neuen DataFrame für die skalierten Daten
df_scaled = pd.DataFrame(scaled_data, columns=columns_to_scale)
df_scaled.head()

,Fe,C,Si,Mn,P,S,Cr,Mo,Ni,Al,...,Ti,V,W,Pb,Sn,B,Ca,Zr,As,Bi
0,0.0,0.032564,0.185649,0.531216,0.028685,0.029073,0.007472,0.005087,0.033725,0.001998,...,0.001000,0.001998,0.039221,0.014889,0.007968,0.001,0.001000,0.001998,0.049457,0.011336
1,0.0,0.150143,0.122218,0.542324,0.021468,0.040278,0.005187,0.004092,0.038259,0.017152,...,0.003195,0.001798,0.039221,0.034112,0.012126,0.001,0.001299,0.001998,0.061001,0.010742
2,0.0,0.134531,0.235072,0.470004,0.018429,0.025765,0.004988,0.004888,0.043730,0.002497,...,0.001199,0.001798,0.039221,0.014889,0.018920,0.001,0.001000,0.001998,0.061847,0.009950
3,0.0,0.234281,0.242162,0.486738,0.012818,0.030723,0.004988,0.004888,0.038836,0.002297,...,0.001199,0.002297,0.039221,0.014889,0.008861,0.001,0.001000,0.001998,0.034691,0.009950
4,0.0,0.227136,0.253091,0.496524,0.011336,0.056097,0.004988,0.003992,0.026934,0.002297,...,0.001399,0.002297,0.039221,0.014889,0.010544,0.001,0.001000,0.001998,0.043443,0.010247


In [3]:
#X_train, X_test = train_test_split(df_scaled,test_size=0.2, random_state=42)

# cntr: Cluster centers
# u: Final fuzzy partition matrix
# u0: Initial guess at fuzzy partition matrix
# d: Final Euclidean distance matrix
# jm: Objective function history
# p: Number of iterations run
# fpc: Fuzzy partition coefficient
#cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(X_train.transpose(),3,1.4, error=0.01, maxiter=10000, init=None)

#Cluster Centers (cntr): This shows the center of each cluster. 
#You can compare these centers to understand how different clusters differ from each other in terms of chemical properties.
#print(cntr)
#Membership Matrix (u): Each element's degree of belonging to each cluster. 
#A higher value indicates a stronger association with that cluster.
#print(u)
#Fuzzy Partition Coefficient (fpc): A value close to 1 indicates well-separated clusters. 
#Lower values indicate overlapping clusters.
#print(fpc)

#u, u0, d, jm, p, fpc = fuzz.cluster.cmeans_predict(df.transpose(), cntr, 5, error=0.01, maxiter=10000)

#print(fpc)

In [4]:
def fuzzy_c_means_cluster(data, labels_df, name_of_label, number_of_cluster):
    
    # Standardizing the Data
    scaler = FunctionTransformer(np.log1p, validate=True)
    data_scaled = scaler.fit_transform(data)
    
    # Apply fuzzy-c-mean with hard parameters
    cntr, u1, u0, d, jm, p, fpc = fuzz.cluster.cmeans(data_scaled.transpose(), number_of_cluster, 2, error=0.005, maxiter=10000, init=None)
    
    # Transpose fuzzy matrix u
    u_df = pd.DataFrame(u1.T)
    
    # Create cluster membership overview
    cluster_df = pd.DataFrame(u_df.values.round(6), columns=[f'cluster {i+1}' for i in range(u_df.shape[1])], index=data.index)
    
    # Assign each data point to the cluster with the highest membership
    max_membership_indices = np.argmax(u1, axis=0)

    # Count the number of data points in each cluster
    cluster_counts = np.bincount(max_membership_indices, minlength=u1.shape[0])

    # Add the cluster_counts as the top row of cluster_df
    cluster_df.loc['Insgesamt'] = cluster_counts

    # Add labels to the cluster_df
    cluster_df[name_of_label] = labels_df[name_of_label]

    print("Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt:", fpc.round(5))
    
    return cluster_df

In [5]:
fuzzy_c_means_cluster(df_scaled,df_test,'ID 1',9)

Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt: 0.23577


,cluster 1,cluster 2,cluster 3,cluster 4,cluster 5,cluster 6,cluster 7,cluster 8,cluster 9,ID 1
0,0.060686,0.109297,0.109155,0.161420,0.104724,0.077488,0.109815,0.109033,0.158383,St 48
1,0.075251,0.107793,0.119043,0.092780,0.066063,0.040669,0.110773,0.119346,0.268282,St 48
2,0.016879,0.101524,0.095303,0.399679,0.106943,0.015409,0.101329,0.094820,0.068114,St 48
3,0.005022,0.253172,0.150922,0.052457,0.146867,0.002990,0.227609,0.147632,0.013329,St 48
4,0.009038,0.193474,0.118942,0.079711,0.281562,0.005237,0.173550,0.116684,0.021805,St 48
...,...,...,...,...,...,...,...,...,...,...
258,0.293171,0.056785,0.061240,0.056929,0.040631,0.217388,0.057759,0.061409,0.154687,St 48-BK?
259,0.214898,0.050079,0.053734,0.053133,0.036610,0.338995,0.050900,0.053868,0.147782,St37-BK?
260,0.214898,0.050079,0.053734,0.053133,0.036610,0.338995,0.050900,0.053868,0.147782,St48-BK?
261,0.318268,0.069903,0.077254,0.061410,0.047859,0.123150,0.071412,0.077563,0.153182,St 48-BK


In [6]:
df_sv = pd.read_csv('labeled_OES_data_proper.csv')

# Auswählen der Spalten, die skaliert werden sollen
columns_to_scale = ['C','Si','Mn','P','S','Cr','Ni','Al','Cu']

# Skalieren der ausgewählten Spalten
scaler = FunctionTransformer(np.log1p, validate=True)
scaled_data = scaler.fit_transform(df_sv[columns_to_scale])

# Erstellen eines neuen DataFrame für die skalierten Daten
df_scaled1 = pd.DataFrame(scaled_data, columns=columns_to_scale)
df_scaled1

,C,Si,Mn,P,S,Cr,Ni,Al,Cu
0,0.254384,0.186341,0.425377,0.024579,0.043618,0.034321,0.048203,0.002081,0.134822
1,0.092837,0.680568,0.515614,0.026090,0.034031,0.009141,0.041702,0.003128,0.180653
2,0.124575,0.606590,0.641678,0.014741,0.046470,0.023196,0.106310,0.053288,0.311643
3,0.118079,0.583425,0.672604,0.014839,0.045531,0.029510,0.106760,0.049044,0.312619
4,0.211187,0.213959,0.547626,0.027309,0.041224,0.014776,0.048668,0.004205,0.154069
...,...,...,...,...,...,...,...,...,...
432,0.153193,0.174163,0.210463,0.116938,0.008481,0.003394,0.025034,0.010363,0.031870
433,0.165345,0.093873,0.233838,0.082520,0.061396,0.007015,0.014731,0.000860,0.047914
434,0.004540,0.121377,0.065694,0.122218,0.049266,0.003344,0.025790,0.005137,0.054346
435,0.061048,0.077979,0.157538,0.122218,0.024644,0.002846,0.048076,0.004440,0.135448


In [7]:
fuzzy_c_means_cluster(df_scaled1,df_sv,'steel_class',5)

Der folgende Wert gibt an wie klar die Cluster getrennt sind, umso näher an 1 desto klarer sind sie getrennt: 0.43672


,cluster 1,cluster 2,cluster 3,cluster 4,cluster 5,steel_class
0,0.358015,0.361656,0.025484,0.243323,0.011522,St 48
1,0.028813,0.028132,0.012616,0.028400,0.902039,St Si
2,0.102486,0.108415,0.045933,0.113782,0.629384,St Si
3,0.109308,0.117209,0.049359,0.124022,0.600101,St Si
4,0.060201,0.186691,0.007074,0.740467,0.005567,St 48
...,...,...,...,...,...,...
433,0.244378,0.185574,0.372007,0.154425,0.043617,Puddelstahl
434,0.217474,0.179910,0.353338,0.159490,0.089789,Puddelstahl
435,0.204981,0.166751,0.426666,0.143778,0.057824,Puddelstahl
436,0.222240,0.182430,0.342979,0.161225,0.091126,Puddelstahl


In [42]:
def fuzzy_c_means_clustering(data,number_of_clusters):
    # Extracting features and class labels
    features = data.iloc[:, :-1].values.T
    labels = data.iloc[:, -1]

    # Perform fuzzy c-means clustering
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        features, number_of_clusters, 2, error=0.005, maxiter=10000, init=None
    )

    # Determining the cluster to which each data point belongs
    cluster_membership = np.argmax(u, axis=0)

    # Creating a dataframe to count occurrences
    cluster_counts = pd.DataFrame(0, index=np.unique(labels), columns=np.arange(number_of_clusters))

    # Counting occurrences
    for label, cluster in zip(labels, cluster_membership):
        cluster_counts.loc[label, cluster] += 1

    return cluster_counts

In [43]:
fuzzy_c_means_clustering(df_sv,5)

,0,1,2,3,4
Flussstahl,0,0,9,0,0
Puddelstahl,0,5,21,0,0
St 37,0,2,91,0,4
St 48,1,91,7,95,80
St Si,31,0,0,0,0
